In [0]:
from pyspark.sql import functions as F

catalog = "adb_ecommerce_de_7405607562556202"

geolocation_df = spark.table(
    f"{catalog}.ecommerce_bronze.geolocation"
)

geolocation_df.printSchema()
geolocation_df.show(5, truncate=False)

In [0]:
geolocation_df.select([
    F.count(F.when(geolocation_df[c].isNull(), 1)).alias(c)
    for c in geolocation_df.columns
]).show()

In [0]:
silver_geolocation = (
    geolocation_df
    .withColumn(
        "geolocation_city",
        F.trim(geolocation_df["geolocation_city"])
    )
    .withColumn(
        "geolocation_state",
        F.trim(geolocation_df["geolocation_state"])
    )
)

In [0]:
silver_path = "abfss://ecommerce@mayankecommerce017.dfs.core.windows.net/silver/geolocation"

silver_geolocation.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_path)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.ecommerce_silver.geolocation
USING DELTA
LOCATION '{silver_path}'
""")

In [0]:
spark.table(
    f"{catalog}.ecommerce_silver.geolocation"
).show(5, truncate=False)